# 03-2. 조건 기반 출시 전 체크리스트 LLM 생성

이 코드는 `02-1`에서 만든 CSV 4개를 불러와서, 개발자가 입력한 조건에 맞는 **출시 전 점검 체크리스트**를 LLM으로 생성한다.

핵심은 **우선순위를 LLM이 새로 판단하지 않도록 제한하는 것**이다.  
우선순위는 `03-1`에서 사전에 계산된 반복 이슈 지표를 기준으로 먼저 정리하고, 이 코드에서는 그 결과를 체크리스트 문장으로 바꾸는 데 LLM을 사용한다.

## 역할

1. 개발자 조건을 입력한다.
2. 조건에 맞는 게임과 반복 이슈 근거를 확인한다.
3. 체크리스트 생성 전에 근거 데이터를 먼저 살펴본다.
4. Steam 태그 DNA 정보를 보조 근거로 확인한다.
5. `03-1`에서 사전에 계산된 상·중·하 우선순위를 고정값으로 사용한다.
6. LLM은 우선순위를 계산하거나 변경하지 않고, 이미 계산된 우선순위와 근거를 바탕으로 체크리스트 문장을 작성한다.
7. LLM 결과가 근거표의 우선순위와 다르게 나오면, 출력 단계에서 근거표의 우선순위로 다시 고정한다.
8. 결과는 저장하지 않고 코드 화면에 표 형태로 출력한다.

## 사용 데이터

- `prelaunch_game_base.csv`
- `prelaunch_issue_repeat_summary.csv`
- `prelaunch_condition_issue_summary.csv`
- `prelaunch_checklist_evidence_base.csv`
- `steam_indie_games_graded.csv`
  - 조원 EDA에서 사용한 성과 등급 기반 Steam 태그 DNA 참고 데이터

## 해석 기준

이 코드는 리뷰 수가 많은 게임에서 많이 나온 이슈보다, **여러 게임에서 반복적으로 나타난 이슈**를 더 중요하게 본다.

다만 이 단계에서 LLM이 우선순위를 새로 정하지는 않는다.  
우선순위는 이전 단계에서 계산된 `priority_level`을 사용하며, LLM은 그 값을 바꾸지 않고 개발자가 이해하기 쉬운 체크리스트 문장으로 정리하는 역할만 한다.

`High urgency`는 이전 LLM 리뷰 분석에서 나온 보조 지표이므로, 우선순위 산정 기준으로 새로 쓰지 않는다.


# 0. 환경설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import re
import ast
import json
import platform
from pathlib import Path
from datetime import datetime
from typing import List, Literal

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# LLM / Pydantic 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 200)

# 1. 기본 설정

In [ ]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break


# ============================================================
# 출시 전 master 데이터 폴더
# ============================================================
PRELAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "prelaunch"
MASTER_DIR = PRELAUNCH_OUTPUT_DIR / "master"

# 03-1에서 생성한 체크리스트 근거 CSV 폴더
DATA_DIR = MASTER_DIR / "prelaunch_checklist_data"

# ============================================================
# 체크리스트 생성용 입력 파일
# ============================================================
GAME_BASE_PATH = DATA_DIR / "prelaunch_game_base.csv"
ISSUE_REPEAT_PATH = DATA_DIR / "prelaunch_issue_repeat_summary.csv"
CONDITION_ISSUE_PATH = DATA_DIR / "prelaunch_condition_issue_summary.csv"
EVIDENCE_BASE_PATH = DATA_DIR / "prelaunch_checklist_evidence_base.csv"

# ============================================================
# 조원 EDA 기반 Steam 태그 DNA 참고 데이터
# ============================================================
GRADED_GAME_PATH = ROOT / "data" / "preprocessed" / "steam_indie_games_graded.csv"

print("ROOT:", ROOT)
print("MASTER_DIR:", MASTER_DIR)
print("DATA_DIR:", DATA_DIR)
print("GAME_BASE_PATH exists:", GAME_BASE_PATH.exists())
print("ISSUE_REPEAT_PATH exists:", ISSUE_REPEAT_PATH.exists())
print("CONDITION_ISSUE_PATH exists:", CONDITION_ISSUE_PATH.exists())
print("EVIDENCE_BASE_PATH exists:", EVIDENCE_BASE_PATH.exists())
print("GRADED_GAME_PATH exists:", GRADED_GAME_PATH.exists())

ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
MASTER_DIR: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master
DATA_DIR: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\prelaunch_checklist_data
GAME_BASE_PATH exists: True
ISSUE_REPEAT_PATH exists: True
CONDITION_ISSUE_PATH exists: True
EVIDENCE_BASE_PATH exists: True
GRADED_GAME_PATH exists: True


In [3]:
# ============================================================
# LLM 실행 여부
# ============================================================
# False: LLM 호출 없이 조건 필터링, 프롬프트, 태그 DNA만 확인한다.
# True : 실제 LLM을 호출해서 체크리스트를 생성한다.

RUN_CHECKLIST_LLM = True

# ============================================================
# 개발자 입력 조건
# ============================================================
# 현재는 테스트/시연용으로 조건을 코드에 직접 입력한다.
# 실제 서비스나 대시보드에서는 사용자가 선택한 장르, 가격대, 태그, 플레이 방식을 이 값으로 전달하면 된다.

USER_CONDITION = {
    "genres": ["Action"],
    "price_group": "10-20",
    "steam_tags": ["Roguelike", "Pixel Graphics"],
    "steam_tag_match": "any",   # "any": 태그 중 하나라도 포함 / "all": 태그를 모두 포함
    "play_style": "Single-player 중심"
}

# ============================================================
# 프롬프트에 넣을 근거 데이터 개수
# ============================================================
TOP_N_EVIDENCE_PER_CONDITION = 8
TOP_N_OVERALL_ISSUES = 8
MAX_EVIDENCE_ROWS_FOR_PROMPT = 40

# ============================================================
# LLM 호출 설정
# ============================================================
MAX_RETRIES = 3
TEMPERATURE = 0.0

print("RUN_CHECKLIST_LLM:", RUN_CHECKLIST_LLM)
print("USER_CONDITION:")
print(json.dumps(USER_CONDITION, ensure_ascii=False, indent=2))

RUN_CHECKLIST_LLM: True
USER_CONDITION:
{
  "genres": [
    "Action"
  ],
  "price_group": "10-20",
  "steam_tags": [
    "Roguelike",
    "Pixel Graphics"
  ],
  "steam_tag_match": "any",
  "play_style": "Single-player 중심"
}


# 2. Vertex AI / PydanticAI 설정

In [ ]:
# ============================================================
# Vertex AI 설정
# ============================================================
# 03_run_llm_allgames_analysi.ipynb와 비슷한 방식으로 구성한다.

load_dotenv()

GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")
GOOGLE_CLOUD_LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")

provider = None
vertex_model = None

if GOOGLE_CLOUD_PROJECT:
    provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )
    vertex_model = GoogleModel(GEMINI_MODEL, provider=provider)

print("GOOGLE_CLOUD_PROJECT:", GOOGLE_CLOUD_PROJECT)
print("GOOGLE_CLOUD_LOCATION:", GOOGLE_CLOUD_LOCATION)
print("GEMINI_MODEL:", GEMINI_MODEL)
print("Vertex model 생성 여부:", "O" if vertex_model is not None else "X")

GOOGLE_CLOUD_PROJECT: gen-lang-client-0587784564
GOOGLE_CLOUD_LOCATION: global
GEMINI_MODEL: gemini-3.1-flash-lite-preview
Vertex model 생성 여부: O


# 3. 공통 함수

In [5]:
# ============================================================
# JSON 변환 보조 함수
# ============================================================

def to_serializable(obj):
    """Pydantic, pandas, numpy 값을 기본 타입으로 바꾼다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


# ============================================================
# 문자열 검색 함수
# ============================================================

def text_contains_any(text, values):
    text = str(text).lower()
    values = [str(v).lower() for v in values if str(v).strip()]
    return any(v in text for v in values)


def text_contains_all(text, values):
    text = str(text).lower()
    values = [str(v).lower() for v in values if str(v).strip()]
    return all(v in text for v in values)


# ============================================================
# Steam 태그 정리 함수
# ============================================================

def normalize_tag_name(tag):
    """Rogue-like, Roguelike처럼 표기가 달라도 비교할 수 있게 간단히 정규화한다."""
    tag = str(tag).lower()
    tag = re.sub(r"[^a-z0-9가-힣]", "", tag)
    return tag


def parse_tag_names(value):
    """tags 컬럼을 태그명 리스트로 바꾼다."""
    if pd.isna(value):
        return []

    text = str(value).strip()
    if text == "":
        return []

    try:
        parsed = json.loads(text.replace("'", '"'))
    except Exception:
        try:
            parsed = ast.literal_eval(text)
        except Exception:
            parsed = None

    if isinstance(parsed, dict):
        return list(parsed.keys())
    if isinstance(parsed, list):
        return [str(x) for x in parsed]

    return [x.strip() for x in text.split(",") if x.strip()]


# ============================================================
# 프롬프트용 텍스트 표 생성 함수
# ============================================================

def df_to_text_table(df, columns, max_rows=20):
    """tabulate 없이 LLM 프롬프트에 넣을 간단 텍스트 표를 만든다."""
    small = df[columns].head(max_rows).copy()

    if len(small) == 0:
        return "해당 조건에 맞는 근거 데이터가 없습니다."

    lines = []
    header = " | ".join(columns)
    lines.append(header)
    lines.append("-" * len(header))

    for _, row in small.iterrows():
        values = [str(row.get(col, "")) for col in columns]
        lines.append(" | ".join(values))

    return "\n".join(lines)


# ============================================================
# 우선순위 정렬 함수
# ============================================================

def add_priority_order(df, priority_col="priority_level"):
    order_map = {"상": 1, "중": 2, "하": 3}
    out = df.copy()
    out["priority_order"] = out[priority_col].map(order_map).fillna(9)
    return out

# 4. 데이터 불러오기

In [6]:
# ============================================================
# 03-1에서 만든 CSV 불러오기
# ============================================================

game_base = pd.read_csv(GAME_BASE_PATH)
issue_repeat_summary = pd.read_csv(ISSUE_REPEAT_PATH)
condition_issue_summary = pd.read_csv(CONDITION_ISSUE_PATH)
evidence_base = pd.read_csv(EVIDENCE_BASE_PATH)

graded_games = pd.read_csv(GRADED_GAME_PATH)

# 문자열 컬럼 정리
for df in [game_base, issue_repeat_summary, condition_issue_summary, evidence_base, graded_games]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

print("game_base:", game_base.shape)
print("issue_repeat_summary:", issue_repeat_summary.shape)
print("condition_issue_summary:", condition_issue_summary.shape)
print("evidence_base:", evidence_base.shape)
print("graded_games:", graded_games.shape)

game_base: (151, 15)
issue_repeat_summary: (21, 15)
condition_issue_summary: (3846, 18)
evidence_base: (3296, 19)
graded_games: (9169, 24)


C:\Users\joon5\AppData\Local\Temp\ipykernel_16252\2581022686.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\joon5\AppData\Local\Temp\ipykernel_16252\2581022686.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

# 5. 사용자 조건에 맞는 게임 확인

In [7]:
# ============================================================
# 사용자 조건에 맞는 게임 필터링
# ============================================================
# 이 단계는 LLM이 근거를 해석하기 전에,
# 입력 조건에 해당하는 게임 수가 충분한지 확인하기 위한 작업이다.

matched_games = game_base.copy()

# 장르 조건
genres = USER_CONDITION.get("genres", [])
if genres:
    matched_games = matched_games[
        matched_games["genres_text"].apply(lambda x: text_contains_any(x, genres))
    ]

# 가격대 조건
price_group = USER_CONDITION.get("price_group")
if price_group:
    matched_games = matched_games[matched_games["price_group"] == price_group]

# Steam 태그 조건
# steam_tag_match는 사용자 조건과 유사 게임 확인에 사용한다.
# 체크리스트 근거는 선택된 태그 각각의 조건별 근거를 함께 가져온다.
steam_tags = USER_CONDITION.get("steam_tags", [])
steam_tag_match = USER_CONDITION.get("steam_tag_match", "any")

if steam_tags:
    if steam_tag_match == "all":
        matched_games = matched_games[
            matched_games["top_steam_tags_text"].apply(lambda x: text_contains_all(x, steam_tags))
        ]
    else:
        matched_games = matched_games[
            matched_games["top_steam_tags_text"].apply(lambda x: text_contains_any(x, steam_tags))
        ]

# 플레이 방식 조건
play_style = USER_CONDITION.get("play_style")
if play_style:
    matched_games = matched_games[matched_games["play_style"] == play_style]

matched_game_count = matched_games["appid"].nunique()
matched_review_count = matched_games["review_count"].sum()

print("조건에 맞는 게임 수:", matched_game_count)
print("조건에 맞는 분석 리뷰 수:", matched_review_count)
print("조건에 맞는 게임 예시")
print(matched_games[["appid", "game_name", "price_group", "play_style", "review_count"]].head(10).to_string(index=False))

조건에 맞는 게임 수: 1
조건에 맞는 분석 리뷰 수: 1
조건에 맞는 게임 예시
  appid         game_name price_group       play_style  review_count
2668540 Targeosity Horror       10-20 Single-player 중심             1


# 6. 조건별 반복 이슈 근거 선택

In [8]:
# ============================================================
# 사용자 조건에 맞는 condition evidence 선택
# ============================================================
# 여기서는 조건별로 미리 만들어둔 근거표를 가져온다.
# 예: genre=Action, price_group=10-20, steam_tag=Roguelike, play_style=Single-player 중심
#
# 중요:
# 이 단계에서 사용하는 priority_level은 LLM이 새로 판단한 값이 아니다.
# 03-1에서 이슈 발생 게임 수, 조건 내 발생 비율, Steam 비추천 맥락을 기준으로 미리 계산된 값이다.
# High urgency는 이전 LLM 분석에서 나온 보조 지표이며, priority_level을 새로 올리는 기준으로 사용하지 않는다.
# 이후 LLM 프롬프트에는 이 값을 fixed_priority로 전달해서 우선순위를 바꾸지 못하게 한다.

selected_parts = []

# 장르 근거
for genre in USER_CONDITION.get("genres", []):
    temp = evidence_base[
        (evidence_base["condition_type"] == "genre")
        & (evidence_base["condition_value"] == genre)
    ].copy()
    selected_parts.append(temp)

# 가격대 근거
price_group = USER_CONDITION.get("price_group")
if price_group:
    temp = evidence_base[
        (evidence_base["condition_type"] == "price_group")
        & (evidence_base["condition_value"] == price_group)
    ].copy()
    selected_parts.append(temp)

# Steam 태그 근거
for steam_tag in USER_CONDITION.get("steam_tags", []):
    temp = evidence_base[
        (evidence_base["condition_type"] == "steam_tag")
        & (evidence_base["condition_value"] == steam_tag)
    ].copy()
    selected_parts.append(temp)

# 플레이 방식 근거
play_style = USER_CONDITION.get("play_style")
if play_style:
    temp = evidence_base[
        (evidence_base["condition_type"] == "play_style")
        & (evidence_base["condition_value"] == play_style)
    ].copy()
    selected_parts.append(temp)

selected_evidence = pd.concat(selected_parts, ignore_index=True)

# 우선순위/발생 비율 기준 정렬
# priority_level은 이미 계산된 우선순위이므로 여기서는 정렬 기준으로만 사용한다.
# High urgency는 LLM 기반 보조 지표이므로, LLM 프롬프트에 들어갈 근거 선별 정렬 기준에서는 제외한다.
selected_evidence = add_priority_order(selected_evidence)
selected_evidence = selected_evidence.sort_values(
    ["priority_order", "issue_game_ratio", "issue_game_count", "negative_game_count", "total_issue_review_count"],
    ascending=[True, False, False, False, False]
)

# 조건별로 너무 많은 근거가 들어가지 않도록 상위 N개만 사용
selected_evidence = (
    selected_evidence
    .groupby(["condition_type", "condition_value"], group_keys=False)
    .head(TOP_N_EVIDENCE_PER_CONDITION)
    .reset_index(drop=True)
)

selected_evidence_for_prompt = selected_evidence.head(MAX_EVIDENCE_ROWS_FOR_PROMPT).copy()
selected_evidence_for_prompt["fixed_priority"] = selected_evidence_for_prompt["priority_level"]
selected_evidence_for_prompt["source_condition"] = (
    selected_evidence_for_prompt["condition_type"].astype(str)
    + "="
    + selected_evidence_for_prompt["condition_value"].astype(str)
)

print("선택된 근거 데이터:", selected_evidence.shape)
print(selected_evidence_for_prompt[[
    "fixed_priority", "issue_direction", "condition_type", "condition_value", "issue_name_kor",
    "condition_game_count", "issue_game_count", "issue_game_ratio",
    "negative_game_count", "high_urgency_game_count"
]].head(20).to_string(index=False))


선택된 근거 데이터: (32, 20)
fixed_priority issue_direction condition_type  condition_value issue_name_kor  condition_game_count  issue_game_count  issue_game_ratio  negative_game_count  high_urgency_game_count
             상           강화 요소    price_group            10-20          긍정 칭찬                    45                43              95.6                   14                       14
             상           강화 요소      steam_tag   Pixel Graphics          긍정 칭찬                    15                14              93.3                    3                        2
             상           강화 요소     play_style Single-player 중심          긍정 칭찬                   124               115              92.7                   23                       26
             상           강화 요소          genre           Action          긍정 칭찬                    62                57              91.9                   14                       13
             상          리스크 요소    price_group            10-20       

# 7. 전체 게임 기준 반복 이슈

In [9]:
# ============================================================
# 전체 게임 기준 반복 이슈
# ============================================================
# 조건별 근거만 보면 특정 조건의 특징인지, 전체적으로 흔한 이슈인지 판단하기 어렵다.
# 그래서 전체 기준선 이슈도 함께 프롬프트에 넣는다.
# High urgency는 LLM 기반 보조 지표이므로, 전체 기준선 이슈 선별 정렬 기준에서는 제외한다.

overall_issues = add_priority_order(issue_repeat_summary)
overall_issues = overall_issues.sort_values(
    ["priority_order", "issue_game_ratio", "issue_game_count", "negative_game_count", "total_issue_review_count"],
    ascending=[True, False, False, False, False]
).head(TOP_N_OVERALL_ISSUES)

print("전체 기준선 이슈")
print(overall_issues[
    ["issue_name_kor", "issue_game_count", "issue_game_ratio", "negative_game_count", "high_urgency_game_count", "priority_level"]
].to_string(index=False))

전체 기준선 이슈
issue_name_kor  issue_game_count  issue_game_ratio  negative_game_count  high_urgency_game_count priority_level
         긍정 칭찬               140              92.7                   28                       31              상
      게임플레이 루프               108              71.5                   63                       49              상
       그래픽/사운드                89              58.9                   39                       32              상
        콘텐츠 분량                88              58.3                   37                       21              상
         UI/UX                86              57.0                   49                       46              상
           난이도                72              47.7                   28                       21              상
         가격/가치                67              44.4                   26                       16              상
            버그                65              43.0                   32                       

# 8. Steam 태그 DNA 참고 정보

## Steam 태그 DNA 참고 정보란?

조원 EDA에서 사용한 관점은 **성과가 좋은 게임들이 어떤 Steam 태그를 함께 가지고 있는지**를 보는 것이다.

이 코드에서는 사용자가 입력한 Steam 태그가 `high_high`, `high_mid`, `mid_high` 같은 성과 상위권 게임에서 얼마나 자주 나타나는지 계산한다.

단, 이 정보는 **이 태그를 넣으면 성공한다**는 뜻이 아니다.  
체크리스트를 만들 때 Steam 태그 조건을 해석하는 **보조 근거**로만 사용한다.


In [10]:
# ============================================================
# Steam 태그 DNA 참고 정보 생성
# ============================================================
# 사용자가 입력한 Steam 태그가 성과 등급이 높은 게임에서 얼마나 나타나는지 확인한다.
# 이 정보는 체크리스트의 직접 근거가 아니라, Steam 태그 조건 해석용 보조 근거다.

graded_tag_base = graded_games[["appid", "name", "performance_grade", "tags"]].copy()
graded_tag_base["tag_list"] = graded_tag_base["tags"].apply(parse_tag_names)

graded_tag_long = graded_tag_base.explode("tag_list").dropna(subset=["tag_list"]).copy()
graded_tag_long["steam_tag"] = graded_tag_long["tag_list"].astype(str).str.strip()
graded_tag_long = graded_tag_long[graded_tag_long["steam_tag"] != ""]
graded_tag_long["tag_norm"] = graded_tag_long["steam_tag"].apply(normalize_tag_name)

TOP_TIER_GRADES = ["high_high", "high_mid", "mid_high"]

tag_dna_rows = []

for input_tag in USER_CONDITION.get("steam_tags", []):
    input_norm = normalize_tag_name(input_tag)
    matched_tag_rows = graded_tag_long[graded_tag_long["tag_norm"] == input_norm].copy()

    total_game_count = matched_tag_rows["appid"].nunique()
    high_high_game_count = matched_tag_rows.loc[matched_tag_rows["performance_grade"] == "high_high", "appid"].nunique()
    high_mid_game_count = matched_tag_rows.loc[matched_tag_rows["performance_grade"] == "high_mid", "appid"].nunique()
    mid_high_game_count = matched_tag_rows.loc[matched_tag_rows["performance_grade"] == "mid_high", "appid"].nunique()
    top_tier_game_count = matched_tag_rows.loc[matched_tag_rows["performance_grade"].isin(TOP_TIER_GRADES), "appid"].nunique()

    top_tier_ratio = 0
    if total_game_count > 0:
        top_tier_ratio = round(top_tier_game_count / total_game_count * 100, 1)

    matched_tag_name = ""
    if len(matched_tag_rows) > 0:
        matched_tag_name = matched_tag_rows["steam_tag"].mode().iloc[0]

    if total_game_count == 0:
        note = f"'{input_tag}' 태그는 성과 등급 데이터에서 확인되지 않았다."
    else:
        note = (
            f"'{input_tag}' 태그는 성과 등급 데이터에서 {total_game_count}개 게임에 나타났고, "
            f"그중 성과 상위권(HH/HM/MH) 게임은 {top_tier_game_count}개({top_tier_ratio}%)다. "
            f"이는 성공 보장이 아니라 태그 조건 해석용 참고 정보다."
        )

    tag_dna_rows.append({
        "input_steam_tag": input_tag,
        "matched_steam_tag": matched_tag_name,
        "total_game_count": total_game_count,
        "high_high_game_count": high_high_game_count,
        "high_mid_game_count": high_mid_game_count,
        "mid_high_game_count": mid_high_game_count,
        "top_tier_game_count": top_tier_game_count,
        "top_tier_ratio": top_tier_ratio,
        "tag_dna_note": note,
    })

tag_dna_summary = pd.DataFrame(tag_dna_rows)

print("Steam 태그 DNA 참고 정보")
print(tag_dna_summary.to_string(index=False))

Steam 태그 DNA 참고 정보
input_steam_tag matched_steam_tag  total_game_count  high_high_game_count  high_mid_game_count  mid_high_game_count  top_tier_game_count  top_tier_ratio                                                                                                           tag_dna_note
      Roguelike        Rogue-like              1136                   149                   29                  267                  445            39.2      'Roguelike' 태그는 성과 등급 데이터에서 1136개 게임에 나타났고, 그중 성과 상위권(HH/HM/MH) 게임은 445개(39.2%)다. 이는 성공 보장이 아니라 태그 조건 해석용 참고 정보다.
 Pixel Graphics    Pixel Graphics              2197                   233                   34                  566                  833            37.9 'Pixel Graphics' 태그는 성과 등급 데이터에서 2197개 게임에 나타났고, 그중 성과 상위권(HH/HM/MH) 게임은 833개(37.9%)다. 이는 성공 보장이 아니라 태그 조건 해석용 참고 정보다.


# 9. LLM 출력 스키마 정의

이 단계에서는 LLM이 반환해야 할 체크리스트 결과 형식을 정의한다.

여기서 중요한 점은 `상·중·하 우선순위`를 LLM이 새로 판단하는 것이 아니라는 점이다.  
우선순위는 앞 단계에서 조건별 반복 이슈 데이터를 기준으로 이미 정리된 값을 사용한다.

따라서 스키마에서 `priority`는 LLM의 판단값이 아니라, 근거표의 `fixed_priority`를 그대로 복사하는 값으로 해석한다.

| 구분 | 역할 |
|---|---|
| 우선순위 판단 | 하지 않음 |
| 우선순위 사용 | 근거표의 `fixed_priority` 값을 그대로 사용 |
| 체크리스트 작성 | 이미 정리된 우선순위와 근거를 질문형 점검 항목으로 문장화 |
| 근거 요약 | 반복 이슈, Steam 비추천 맥락, High urgency 보조 분포, 태그 DNA 참고 정보를 사람이 읽기 쉽게 정리 |
| 제한 사항 | 근거표에 없는 이슈를 새로 만들지 않음 |

`high_urgency_game_count`는 이전 리뷰 분석 단계에서 LLM이 분류한 urgency 결과를 집계한 값이다.  
따라서 객관적인 장애 지표가 아니며, `03-1`에서 이미 계산된 priority를 바꾸는 데 사용하지 않는다.  
이 코드에서는 사람이 해석할 때 참고할 보조 정보로만 사용한다.

In [11]:
# ============================================================
# 체크리스트 출력 스키마
# ============================================================
# 표로 출력하기 쉽도록 필드명을 체크리스트 표 형태에 맞춘다.
# priority는 LLM이 새로 판단하는 값이 아니라, 근거표의 fixed_priority를 복사하는 값이다.

class ChecklistItem(BaseModel):
    priority: Literal["상", "중", "하"] = Field(description="근거표의 fixed_priority를 그대로 복사한 고정 점검 중요도")
    issue_direction: Literal["강화 요소", "리스크 요소", "참고 요소"] = Field(description="근거표의 issue_direction 값을 그대로 복사한 이슈 해석 방향")
    category: str = Field(description="점검 구분. 예: 기술 안정성, 조작감, 핵심 루프, 난이도, UI/UX, 콘텐츠 분량, 그래픽/사운드, 가격 대비 가치")
    check_question: str = Field(description="출시 전 확인해야 할 질문형 체크 항목. 반드시 질문형으로 작성")
    evidence_issue: str = Field(description="근거가 된 이슈명. 반드시 근거표의 issue_name_kor 값 중 하나를 그대로 사용")
    evidence_summary: str = Field(description="제공된 근거표의 수치와 조건을 바탕으로 한 요약")
    how_to_check: str = Field(description="개발자가 출시 전에 확인할 수 있는 방법")
    source_conditions: List[str] = Field(description="이 근거가 나온 조건 목록")


class PrelaunchChecklistResult(BaseModel):
    title: str = Field(description="체크리스트 제목")
    condition_summary: str = Field(description="사용자 입력 조건 요약")
    data_summary: str = Field(description="근거 데이터 규모 요약")
    tag_dna_summary: str = Field(description="Steam 태그 DNA 참고 정보 요약")
    high_priority: List[ChecklistItem] = Field(description="fixed_priority가 상인 체크리스트")
    mid_priority: List[ChecklistItem] = Field(description="fixed_priority가 중인 체크리스트")
    low_priority: List[ChecklistItem] = Field(description="fixed_priority가 하인 체크리스트")
    cautions: List[str] = Field(description="해석 시 주의사항")
    final_summary: str = Field(description="전체 요약")


print("체크리스트 출력 스키마 정의 완료")


체크리스트 출력 스키마 정의 완료


# 10. LLM 프롬프트 생성

이 단계에서는 LLM에게 전달할 프롬프트를 만든다.

프롬프트에는 개발자 입력 조건, 조건에 맞는 게임 수, 조건별 반복 이슈 근거표, 전체 게임 기준 반복 이슈, Steam 태그 DNA 참고 정보가 들어간다.

이때 LLM에게 중요한 제약을 준다.

| 제약 | 내용 |
|---|---|
| 우선순위 재판단 금지 | LLM은 상·중·하 우선순위를 새로 판단하거나 바꾸지 않는다. |
| 고정 우선순위 유지 | 근거표의 `fixed_priority`를 그대로 사용한다. |
| 버킷 이동 금지 | `fixed_priority=상`은 high, `중`은 mid, `하`는 low에 넣는다. |
| 문장화 역할 | LLM은 이미 정리된 우선순위와 근거를 개발자가 이해하기 쉬운 체크리스트 질문으로 바꾼다. |
| 근거 없는 이슈 생성 금지 | 근거표에 없는 이슈명을 새로 만들지 않는다. |
| High urgency 제한 | High urgency는 이전 LLM 분류 결과의 집계값이므로 보조 근거로만 사용한다. |
| 태그 DNA 제한 | Steam 태그 DNA는 성공 예측이 아니라 태그 해석용 보조 근거로만 사용한다. |
| 출력 변동 최소화 | temperature를 0.0으로 설정해 같은 근거에서 결과가 흔들리는 것을 줄인다. |

즉, 이 단계의 LLM은 **우선순위 판단자**가 아니라 **고정된 우선순위를 문장화하는 체크리스트 작성 보조자**이다.


In [12]:
# ============================================================
# 프롬프트 생성 함수
# ============================================================

def build_checklist_prompt(
    user_condition,
    matched_game_count,
    matched_review_count,
    selected_evidence_df,
    overall_issues_df,
    tag_dna_df,
):
    selected_evidence_df = selected_evidence_df.copy()

    if "fixed_priority" not in selected_evidence_df.columns:
        selected_evidence_df["fixed_priority"] = selected_evidence_df["priority_level"]

    if "source_condition" not in selected_evidence_df.columns:
        selected_evidence_df["source_condition"] = (
            selected_evidence_df["condition_type"].astype(str)
            + "="
            + selected_evidence_df["condition_value"].astype(str)
        )

    evidence_cols = [
        "fixed_priority",
        "source_condition",
        "condition_type",
        "condition_value",
        "issue_name_kor",
        "issue_direction",
        "condition_game_count",
        "issue_game_count",
        "issue_game_ratio",
        "positive_game_count",
        "negative_game_count",
        "tag_negative_game_count",
        "high_urgency_game_count",
        "llm_evidence_text",
    ]
    evidence_cols = [col for col in evidence_cols if col in selected_evidence_df.columns]

    overall_cols = [
        "issue_name_kor",
        "issue_game_count",
        "issue_game_ratio",
        "negative_game_count",
        "tag_negative_game_count",
        "high_urgency_game_count",
        "priority_level",
    ]
    overall_cols = [col for col in overall_cols if col in overall_issues_df.columns]

    tag_dna_cols = [
        "input_steam_tag",
        "matched_steam_tag",
        "total_game_count",
        "top_tier_game_count",
        "top_tier_ratio",
        "tag_dna_note",
    ]

    evidence_text = df_to_text_table(
        selected_evidence_df,
        columns=evidence_cols,
        max_rows=MAX_EVIDENCE_ROWS_FOR_PROMPT,
    )

    overall_text = df_to_text_table(
        overall_issues_df,
        columns=overall_cols,
        max_rows=TOP_N_OVERALL_ISSUES,
    )

    tag_dna_text = df_to_text_table(
        tag_dna_df,
        columns=tag_dna_cols,
        max_rows=20,
    )

    prompt = f"""
당신은 Steam 인디게임 출시 전 점검 체크리스트를 작성하는 데이터 분석 보조자입니다.

목표:
개발자가 입력한 장르·가격대·Steam 태그·플레이 방식 조건을 바탕으로,
기존 Steam 인디게임의 D0-D30 초기 리뷰 분석 결과에서 반복된 이슈를 참고하여
출시 전 체크리스트를 작성하세요.

가장 중요한 제한:
- 당신은 우선순위를 새로 계산하거나 판단하지 않습니다.
- 조건별 반복 이슈 근거표의 fixed_priority는 이미 데이터 기준으로 계산된 고정 우선순위입니다.
- 각 체크리스트 항목의 priority는 반드시 근거표의 fixed_priority를 그대로 사용하세요.
- fixed_priority가 "상"인 항목은 high_priority에, "중"인 항목은 mid_priority에, "하"인 항목은 low_priority에 넣으세요.
- fixed_priority를 올리거나 내리거나, 다른 우선순위로 재배치하지 마세요.
- 근거표에 없는 issue_name_kor를 새로 만들지 마세요.
- evidence_issue에는 반드시 조건별 반복 이슈 근거표에 있는 issue_name_kor 값을 그대로 작성하세요.
- issue_direction에는 반드시 근거표의 issue_direction 값을 그대로 작성하세요.

이슈 해석 방향:
- issue_direction이 "리스크 요소"인 경우, 출시 전 문제가 발생하지 않도록 점검하는 질문으로 작성하세요.
- issue_direction이 "강화 요소"인 경우, 이미 유저가 긍정적으로 평가한 요소를 유지하거나 강화하는 질문으로 작성하세요.
- issue_direction이 "참고 요소"인 경우, 단정하지 말고 참고 점검 항목으로 작성하세요.
- 모든 항목을 문제처럼 표현하지 마세요.

근거 사용 기준:
- issue_game_count와 issue_game_ratio는 여러 게임에서 반복되었는지 확인하는 핵심 근거입니다.
- negative_game_count는 Steam 비추천 맥락에서 반복되었는지 확인하는 핵심 근거입니다.
- tag_negative_game_count가 제공되는 경우, 이는 LLM이 이슈 단위로 부정 맥락을 분류한 보조 지표입니다.
- high_urgency_game_count는 이전 LLM 리뷰 분류 결과를 집계한 보조 지표입니다.
- high_urgency_game_count만으로 우선순위를 바꾸거나 새로 판단하지 마세요.
- 태그 수나 리뷰 수만으로 과도하게 단정하지 마세요.
- 근거가 약하면 "참고 수준" 또는 "주의해서 해석"이라고 표현하세요.
- 체크 질문은 반드시 실제 출시 전에 확인 가능한 질문형 문장으로 작성하세요.
- 예: "크래시, 저장 오류, 진행 불가 버그가 없는가?"
- 예: "초반 30분 안에 게임의 핵심 재미가 드러나는가?"
- 예: "이동, 공격, 회피, 상호작용이 즉각적으로 반응하는가?"

Steam 태그 DNA 참고 기준:
- Steam 태그 DNA는 조원 EDA에서 가져온 보조 관점입니다.
- 성과 상위권 게임에서 특정 Steam 태그가 얼마나 자주 나타나는지 보여줍니다.
- 이것은 "해당 태그를 넣으면 성공한다"는 뜻이 아닙니다.
- 체크리스트의 직접 근거는 조건별 반복 이슈 근거표입니다.
- Steam 태그 DNA는 Steam 태그 조건을 해석하는 보조 정보로만 사용하세요.

사용자 입력 조건:
{json.dumps(user_condition, ensure_ascii=False, indent=2)}

조건에 맞는 게임 수:
- 게임 수: {matched_game_count}
- 분석 리뷰 수: {matched_review_count}

조건별 반복 이슈 근거표:
{evidence_text}

전체 게임 기준 반복 이슈:
{overall_text}

Steam 태그 DNA 참고 정보:
{tag_dna_text}

출력 요구:
1. 사용자의 조건을 간단히 요약하세요.
2. 데이터 규모를 간단히 설명하세요.
3. Steam 태그 DNA 참고 정보를 1~2문장으로 요약하세요.
4. high_priority에는 fixed_priority가 "상"인 근거 이슈만 작성하세요.
5. mid_priority에는 fixed_priority가 "중"인 근거 이슈만 작성하세요.
6. low_priority에는 fixed_priority가 "하"인 근거 이슈만 작성하세요.
7. 각 항목은 issue_direction, category, check_question, evidence_issue, evidence_summary, how_to_check가 표로 출력되기 좋은 내용이어야 합니다.
8. evidence_summary에는 가능한 한 issue_game_count, issue_game_ratio, negative_game_count 중 2개 이상을 포함하고, high_urgency_game_count는 보조 정보로만 언급하세요.
9. 마지막에는 해석 시 주의사항을 작성하세요.
"""

    return prompt.strip()


checklist_prompt = build_checklist_prompt(
    user_condition=USER_CONDITION,
    matched_game_count=matched_game_count,
    matched_review_count=matched_review_count,
    selected_evidence_df=selected_evidence_for_prompt,
    overall_issues_df=overall_issues,
    tag_dna_df=tag_dna_summary,
)

print("프롬프트 생성 완료")
print(checklist_prompt[:2500])


프롬프트 생성 완료
당신은 Steam 인디게임 출시 전 점검 체크리스트를 작성하는 데이터 분석 보조자입니다.

목표:
개발자가 입력한 장르·가격대·Steam 태그·플레이 방식 조건을 바탕으로,
기존 Steam 인디게임의 D0-D30 초기 리뷰 분석 결과에서 반복된 이슈를 참고하여
출시 전 체크리스트를 작성하세요.

가장 중요한 제한:
- 당신은 우선순위를 새로 계산하거나 판단하지 않습니다.
- 조건별 반복 이슈 근거표의 fixed_priority는 이미 데이터 기준으로 계산된 고정 우선순위입니다.
- 각 체크리스트 항목의 priority는 반드시 근거표의 fixed_priority를 그대로 사용하세요.
- fixed_priority가 "상"인 항목은 high_priority에, "중"인 항목은 mid_priority에, "하"인 항목은 low_priority에 넣으세요.
- fixed_priority를 올리거나 내리거나, 다른 우선순위로 재배치하지 마세요.
- 근거표에 없는 issue_name_kor를 새로 만들지 마세요.
- evidence_issue에는 반드시 조건별 반복 이슈 근거표에 있는 issue_name_kor 값을 그대로 작성하세요.
- issue_direction에는 반드시 근거표의 issue_direction 값을 그대로 작성하세요.

이슈 해석 방향:
- issue_direction이 "리스크 요소"인 경우, 출시 전 문제가 발생하지 않도록 점검하는 질문으로 작성하세요.
- issue_direction이 "강화 요소"인 경우, 이미 유저가 긍정적으로 평가한 요소를 유지하거나 강화하는 질문으로 작성하세요.
- issue_direction이 "참고 요소"인 경우, 단정하지 말고 참고 점검 항목으로 작성하세요.
- 모든 항목을 문제처럼 표현하지 마세요.

근거 사용 기준:
- issue_game_count와 issue_game_ratio는 여러 게임에서 반복되었는지 확인하는 핵심 근거입니다.
- negative_game_count는 Steam 비추

# 11. LLM 호출 전 확인

실제 LLM을 호출하기 전에, 조건에 맞는 근거 데이터와 프롬프트가 제대로 만들어졌는지 확인한다.

이 확인 단계에서 봐야 할 핵심은 다음과 같다.

| 확인 항목 | 의미 |
|---|---|
| 조건에 맞는 게임 수 | 입력 조건의 근거가 충분한지 확인 |
| 선택된 근거 데이터 수 | 체크리스트 생성에 사용할 반복 이슈 근거가 충분한지 확인 |
| Steam 태그 DNA 행 수 | 입력 태그에 대한 보조 참고 정보가 있는지 확인 |
| 프롬프트 내용 | LLM이 우선순위를 새로 판단하지 않고, 제공된 우선순위를 유지하도록 지시되어 있는지 확인 |


In [13]:
# ============================================================
# LLM 호출 전 확인용 요약
# ============================================================
# RUN_CHECKLIST_LLM=False 상태에서도 조건, 근거, 태그 DNA가 잘 잡혔는지 확인한다.

print("조건에 맞는 게임 수:", matched_game_count)
print("조건에 맞는 분석 리뷰 수:", matched_review_count)
print()

print("선택된 근거 데이터 수:", len(selected_evidence))
check_cols = [
    "priority_level", "issue_direction", "condition_type", "condition_value", "issue_name_kor",
    "condition_game_count", "issue_game_count", "issue_game_ratio",
    "negative_game_count", "tag_negative_game_count", "high_urgency_game_count"
]
check_cols = [col for col in check_cols if col in selected_evidence.columns]
print(selected_evidence[check_cols].head(20).to_string(index=False))
print()

print("Steam 태그 DNA 참고 정보")
print(tag_dna_summary.to_string(index=False))
print()

print("프롬프트 앞부분")
print(checklist_prompt[:2500])

조건에 맞는 게임 수: 1
조건에 맞는 분석 리뷰 수: 1

선택된 근거 데이터 수: 32
priority_level issue_direction condition_type  condition_value issue_name_kor  condition_game_count  issue_game_count  issue_game_ratio  negative_game_count  tag_negative_game_count  high_urgency_game_count
             상           강화 요소    price_group            10-20          긍정 칭찬                    45                43              95.6                   14                        0                       14
             상           강화 요소      steam_tag   Pixel Graphics          긍정 칭찬                    15                14              93.3                    3                        0                        2
             상           강화 요소     play_style Single-player 중심          긍정 칭찬                   124               115              92.7                   23                        0                       26
             상           강화 요소          genre           Action          긍정 칭찬                    62                57    

# 12. PydanticAI Agent 설정

이 단계에서는 체크리스트 생성을 담당할 LLM Agent를 설정한다.

Agent의 역할은 다음과 같이 제한한다.

| 구분 | 설명 |
|---|---|
| 하지 않는 일 | 상·중·하 우선순위를 새로 판단하거나 변경하지 않는다. |
| 하지 않는 일 | 근거표에 없는 이슈를 새로 만들지 않는다. |
| 하는 일 | 제공된 우선순위와 근거를 바탕으로 체크리스트 문장을 작성한다. |
| 근거 사용 방식 | 반복 이슈 근거표를 중심으로 설명하고, Steam 태그 DNA는 보조 정보로만 사용한다. |
| High urgency 사용 방식 | 이전 LLM 리뷰 분류 결과의 집계값으로 보고, 보조 근거로만 사용한다. |
| 출력 방식 | 개발자가 출시 전에 확인할 수 있는 질문형 체크리스트로 정리한다. |

즉, Agent는 **판단자**가 아니라 **정리자**로 사용한다.


In [14]:
# ============================================================
# 체크리스트 생성 Agent
# ============================================================

system_prompt = """
당신은 Steam 인디게임 출시 전 체크리스트를 작성하는 데이터 분석 보조자입니다.

당신의 역할은 우선순위 판단이 아니라 문장화입니다.
제공된 근거표의 fixed_priority를 반드시 그대로 사용하세요.
fixed_priority를 새로 계산하거나, 올리거나, 내리거나, 다른 우선순위 목록으로 옮기지 마세요.
근거표에 없는 issue_name_kor를 새로 만들지 마세요.
evidence_issue에는 근거표의 issue_name_kor 값을 그대로 작성하세요.
negative_game_count는 Steam 비추천 맥락 기준의 핵심 근거로 사용하세요.
High urgency 관련 값은 이전 LLM 리뷰 분류 결과를 집계한 보조 지표이므로, 단독 우선순위 기준으로 사용하지 마세요.
Steam 태그 DNA는 보조 근거로만 사용하고, 성공을 보장하는 표현을 쓰지 마세요.
체크리스트는 개발자가 출시 전에 실제로 확인할 수 있는 질문형 문장으로 작성하세요.
"""

checklist_settings = GoogleModelSettings(
    temperature=TEMPERATURE,
)

if vertex_model is not None:
    checklist_agent = Agent(
        vertex_model,
        output_type=PrelaunchChecklistResult,
        system_prompt=system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    checklist_agent = None

print("PydanticAI Agent 생성 여부:", "O" if checklist_agent is not None else "X")


PydanticAI Agent 생성 여부: O


# 13. LLM 체크리스트 생성

이 단계에서는 앞에서 만든 프롬프트를 LLM에 전달해 출시 전 체크리스트를 생성한다.

중요한 점은 다음과 같다.

| 구분 | 설명 |
|---|---|
| 우선순위 산정 | 이 단계에서 LLM이 새로 하지 않는다. |
| 우선순위 기준 | `03-1`에서 조건별 반복 이슈 데이터를 바탕으로 이미 정리된 값을 사용한다. |
| 고정 우선순위 | 프롬프트에는 `fixed_priority`로 전달한다. |
| LLM 역할 | 이미 계산된 우선순위와 근거를 바탕으로 체크리스트 문장을 작성한다. |
| 시급도 사용 | High urgency는 이전 LLM 리뷰 분류 결과의 집계값이므로 보조 근거로만 사용한다. |
| 태그 DNA 사용 | Steam 태그 조건 해석을 돕는 보조 참고 정보로만 사용한다. |
| 출력 검증 | LLM 결과가 근거표와 다른 우선순위를 쓰면, 최종 표에서는 근거표의 고정 우선순위를 다시 적용한다. |

따라서 이 단계의 결과는 “LLM이 우선순위를 새로 판단한 결과”가 아니라,  
**사전에 정리된 우선순위와 근거를 LLM이 문장화한 체크리스트**로 해석한다.


In [15]:
# ============================================================
# LLM 체크리스트 생성
# ============================================================
# RUN_CHECKLIST_LLM=False이면 실제 LLM 호출은 하지 않는다.

if RUN_CHECKLIST_LLM:
    if checklist_agent is None:
        raise RuntimeError(
            "checklist_agent가 생성되지 않았습니다. "
            ".env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
        )

    result = await checklist_agent.run(
        checklist_prompt,
        model_settings=checklist_settings,
    )

    checklist_output = result.output
    checklist_result_dict = to_serializable(checklist_output)

    print("LLM 체크리스트 생성 완료")

else:
    checklist_output = None
    checklist_result_dict = None

    print("RUN_CHECKLIST_LLM=False")
    print("LLM 호출은 하지 않고, 프롬프트와 근거 데이터만 생성했습니다.")

LLM 체크리스트 생성 완료


# 14. LLM 결과 검증 및 표 생성

In [16]:
# ============================================================
# LLM 결과 검증 및 표 생성
# ============================================================

def get_check_category(issue_name, checklist_text=""):
    text = f"{issue_name} {checklist_text}".lower()

    if any(word in text for word in ["버그", "크래시", "안정", "저장", "진행 불가"]):
        return "기술 안정성"
    elif any(word in text for word in ["조작", "input", "반응", "컨트롤"]):
        return "조작감"
    elif any(word in text for word in ["게임플레이", "루프", "핵심 재미"]):
        return "핵심 루프"
    elif "난이도" in text:
        return "난이도"
    elif "밸런스" in text:
        return "밸런스"
    elif any(word in text for word in ["ui", "ux", "튜토리얼", "목표 안내", "메뉴"]):
        return "UI/UX"
    elif any(word in text for word in ["콘텐츠", "분량", "볼륨"]):
        return "콘텐츠 분량"
    elif any(word in text for word in ["그래픽", "사운드", "음악", "비주얼"]):
        return "그래픽/사운드"
    elif any(word in text for word in ["가격", "가치", "price", "value"]):
        return "가격 대비 가치"
    else:
        return "기타"


def build_fixed_priority_maps(evidence_df):
    """
    근거표의 issue_name_kor 기준으로 고정 우선순위와 출처 조건을 만든다.
    """
    priority_order_map = {"상": 1, "중": 2, "하": 3}

    work = evidence_df.copy()

    if "fixed_priority" not in work.columns:
        work["fixed_priority"] = work["priority_level"]

    if "source_condition" not in work.columns:
        work["source_condition"] = (
            work["condition_type"].astype(str)
            + "="
            + work["condition_value"].astype(str)
        )

    work["priority_order"] = work["fixed_priority"].map(priority_order_map).fillna(9)

    sort_cols = [
        "issue_name_kor",
        "priority_order",
        "issue_game_ratio",
        "issue_game_count",
        "negative_game_count",
        "total_issue_review_count",
    ]
    sort_cols = [col for col in sort_cols if col in work.columns]

    ascending_map = {
        "issue_name_kor": True,
        "priority_order": True,
        "issue_game_ratio": False,
        "issue_game_count": False,
        "negative_game_count": False,
        "total_issue_review_count": False,
    }
    ascending = [ascending_map[col] for col in sort_cols]

    fixed_priority_map = (
        work
        .sort_values(sort_cols, ascending=ascending)
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["fixed_priority"]
        .to_dict()
    )

    source_condition_map = (
        work
        .groupby("issue_name_kor")["source_condition"]
        .apply(lambda x: sorted(set([str(v) for v in x if str(v).strip()])))
        .to_dict()
    )

    return fixed_priority_map, source_condition_map


def validate_checklist_priority(result_dict, evidence_df):
    """
    LLM 출력이 근거표의 고정 우선순위와 다르게 나왔는지 확인한다.
    """
    fixed_priority_map, _ = build_fixed_priority_maps(evidence_df)

    priority_info = [
        ("상", "high_priority"),
        ("중", "mid_priority"),
        ("하", "low_priority"),
    ]

    warnings = []

    for bucket_priority, key in priority_info:
        items = result_dict.get(key, [])

        for item in items:
            issue_name = item.get("evidence_issue", item.get("issue_name", ""))
            item_priority = item.get("priority", "")
            fixed_priority = fixed_priority_map.get(issue_name)

            if fixed_priority is None:
                warnings.append(f"근거표에 없는 이슈가 LLM 결과에 포함됨: {issue_name}")
                continue

            if item_priority and item_priority != fixed_priority:
                warnings.append(
                    f"이슈 '{issue_name}'의 LLM priority={item_priority}, 근거표 fixed_priority={fixed_priority}"
                )

            if bucket_priority != fixed_priority:
                warnings.append(
                    f"이슈 '{issue_name}'가 {bucket_priority} 목록에 들어갔지만, 근거표 fixed_priority는 {fixed_priority}"
                )

    return warnings


def make_checklist_table(result_dict, evidence_df=None, drop_unknown_issues=True):
    """
    LLM 체크리스트 결과를 최종 출력용 DataFrame으로 변환한다.
    """
    rows = []

    priority_info = [
        ("상", "high_priority"),
        ("중", "mid_priority"),
        ("하", "low_priority"),
    ]

    fixed_priority_map = {}
    source_condition_map = {}

    if evidence_df is not None:
        fixed_priority_map, source_condition_map = build_fixed_priority_maps(evidence_df)

    for bucket_priority, key in priority_info:
        items = result_dict.get(key, [])

        for item in items:
            issue_name = item.get("evidence_issue", item.get("issue_name", ""))
            check_question = item.get("check_question", item.get("checklist_item", ""))
            category = item.get("category", "")

            if category == "":
                category = get_check_category(issue_name, check_question)

            # 최종 출력 우선순위는 LLM 출력값이 아니라 근거표의 fixed_priority를 사용한다.
            fixed_priority = fixed_priority_map.get(issue_name, bucket_priority)

            # 근거표에 없는 이슈는 제외한다.
            if evidence_df is not None and issue_name not in fixed_priority_map and drop_unknown_issues:
                continue

            rows.append({
                "해석 방향": item.get("issue_direction", ""),
                "구분": category,
                "체크 질문": check_question,
                "우선순위": fixed_priority,
                "근거 이슈": issue_name,
                "근거": item.get("evidence_summary", ""),
                "확인 방법": item.get("how_to_check", item.get("recommended_action", "")),
                "근거 조건": ", ".join(source_condition_map.get(issue_name, [])),
            })

    checklist_df = pd.DataFrame(rows)

    if len(checklist_df) == 0:
        return checklist_df

    priority_order_map = {"상": 1, "중": 2, "하": 3}
    checklist_df["priority_order"] = checklist_df["우선순위"].map(priority_order_map).fillna(9)

    checklist_df = (
        checklist_df
        .sort_values(["priority_order", "구분", "근거 이슈"])
        .drop(columns="priority_order")
    )

    return checklist_df[
        ["해석 방향", "구분", "체크 질문", "우선순위", "근거 이슈", "근거", "확인 방법", "근거 조건"]
    ]


print("LLM 결과 검증 및 체크리스트 표 생성 함수 정의 완료")

LLM 결과 검증 및 체크리스트 표 생성 함수 정의 완료


# 15. 최종 출력 공통 함수

In [17]:
# ============================================================
# 공통 출력 함수 - 간결한 리포트 버전
# ============================================================

def display_block_title(title):
    display(Markdown(f"## {title}"))


def _safe_int(value, default=0):
    try:
        if pd.isna(value):
            return default
        return int(value)
    except Exception:
        return default


def _shorten_for_display(value, max_chars=90):
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\n", " ").strip()

    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."

    return text


def make_part2_intro_markdown():
    genres = ", ".join(USER_CONDITION.get("genres", []))
    price_group = USER_CONDITION.get("price_group", "")
    steam_tags = ", ".join(USER_CONDITION.get("steam_tags", []))
    play_style = USER_CONDITION.get("play_style", "")

    return f"""
## 1. 출시 전 LLM 체크리스트 생성

개발자가 입력한 조건과 유사한 Steam 인디게임의 출시 초기 D0-D30 리뷰를 바탕으로,  
출시 전 개발 단계에서 점검해야 할 항목을 체크리스트로 정리한다.

| 항목 | 값 |
|---|---|
| 입력 장르 | {genres} |
| 입력 가격대 | {price_group} |
| 입력 Steam 태그 | {steam_tags} |
| 입력 플레이 방식 | {play_style} |
| 입력 조건 직접 매칭 게임 수 | {matched_game_count:,}개 |
| 입력 조건 직접 매칭 리뷰 수 | {matched_review_count:,}개 |

체크리스트 생성에는 직접 매칭 결과뿐 아니라,  
장르·가격대·Steam 태그·플레이 방식별로 03-1에서 계산한 반복 이슈 근거도 함께 사용하였다.
""".strip()


def make_prelaunch_criteria_markdown():
    return """
## 2. 체크리스트 판단 기준

- **LLM은 우선순위를 직접 정하지 않는다.**
- 체크리스트 우선순위는 03-1에서 계산한 `priority_level`을 `fixed_priority`로 전달해 그대로 사용한다.
- 우선순위는 이슈 발생 게임 수, 조건 내 발생 비율, Steam 비추천 맥락 발생 게임 수를 기준으로 계산한 값이다.
- `High urgency`는 이전 LLM 리뷰 분류 결과에서 나온 보조 지표이며, 우선순위 산정 기준으로 직접 사용하지 않는다.
- LLM은 계산된 근거를 바탕으로 체크 질문과 확인 방법을 문장화하는 역할만 한다.
""".strip()


def make_compact_prelaunch_evidence_display_df(evidence_df, max_rows=8):
    # 발표/보고서용 근거표는 너무 길게 보여주지 않는다.
    display_cols = [
        "issue_name_kor",
        "issue_direction",
        "fixed_priority",
        "condition_type",
        "condition_value",
        "condition_game_count",
        "issue_game_count",
        "issue_game_ratio",
        "negative_game_count",
        "tag_negative_game_count",
        "high_urgency_game_count",
        "llm_evidence_text",
    ]

    display_cols = [col for col in display_cols if col in evidence_df.columns]

    rename_map = {
        "issue_name_kor": "이슈",
        "issue_direction": "해석 방향",
        "fixed_priority": "우선순위",
        "condition_type": "조건 유형",
        "condition_value": "조건 값",
        "condition_game_count": "조건 내 게임 수",
        "issue_game_count": "이슈 발생 게임 수",
        "issue_game_ratio": "조건 내 발생 비율",
        "negative_game_count": "Steam 비추천 맥락 게임 수",
        "tag_negative_game_count": "LLM 부정 맥락 게임 수",
        "high_urgency_game_count": "High urgency 게임 수",
        "llm_evidence_text": "근거 요약",
    }

    out = evidence_df[display_cols].head(max_rows).copy()

    if "issue_game_ratio" in out.columns:
        out["issue_game_ratio"] = out["issue_game_ratio"].apply(
            lambda x: f"{x:.1%}" if pd.notna(x) and isinstance(x, (int, float)) else x
        )

    if "llm_evidence_text" in out.columns:
        out["llm_evidence_text"] = out["llm_evidence_text"].apply(
            lambda x: _shorten_for_display(x, 90)
        )

    return out.rename(columns=rename_map)


def make_prelaunch_validation_result_markdown(result_dict, evidence_df):
    validation_warnings = validate_checklist_priority(result_dict, evidence_df)

    if len(validation_warnings) == 0:
        return """
## 4. LLM 출력 검증 결과

- 검증 결과: **통과**
- LLM이 출력한 체크리스트 항목의 우선순위가 03-1 근거표의 `fixed_priority`와 일치한다.
- 최종 표에서도 근거표 기준 우선순위를 그대로 사용한다.
""".strip()

    warning_text = "\n".join([f"- {x}" for x in validation_warnings[:5]])

    return f"""
## 4. LLM 출력 검증 결과

- 검증 결과: **보정 필요**
- LLM 출력 일부가 근거표와 다르게 나왔기 때문에, 최종 표에서는 03-1 근거표의 `fixed_priority`를 다시 적용한다.

{warning_text}
""".strip()


def display_common_prelaunch_output_blocks(result_dict, evidence_df, max_evidence_rows=8):
    display(Markdown(make_part2_intro_markdown()))
    display(Markdown(make_prelaunch_criteria_markdown()))

    display_block_title("3. 조건별 반복 이슈 근거표")
    display(Markdown("아래 표는 03-1에서 계산한 근거 중 체크리스트 생성에 사용한 핵심 이슈만 간단히 정리한 것이다."))
    display(make_compact_prelaunch_evidence_display_df(evidence_df, max_rows=max_evidence_rows))

    display(Markdown(make_prelaunch_validation_result_markdown(result_dict, evidence_df)))


print("간결한 리포트용 공통 출력 함수 정의 완료")

간결한 리포트용 공통 출력 함수 정의 완료


# 16. 리포트 형식 출시 전 체크리스트 출력

In [18]:
# ============================================================
# 리포트 형식 출시 전 체크리스트 출력
# ============================================================
def make_styled_checklist_table(checklist_df):
    # 체크리스트 DataFrame을 보고서용 스타일 표로 변환한다.
    return (
        checklist_df
        .style
        .hide(axis="index")
        .set_properties(**{
            "text-align": "left",
            "white-space": "pre-wrap",
            "vertical-align": "top",
            "font-size": "13px",
            "line-height": "1.5",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("font-weight", "bold"),
                    ("background-color", "#f2f2f2"),
                    ("border", "1px solid #cccccc"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #dddddd"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
        ])
    )


def display_checklist_cautions(result_dict):
    # LLM 결과에 포함된 해석 시 주의사항을 출력한다.
    cautions = result_dict.get("cautions", [])

    if cautions:
        display(Markdown("## 6. 해석 시 주의사항"))
        for caution in cautions:
            display(Markdown(f"- {caution}"))


def display_checklist_final_summary(result_dict):
    # LLM 결과에 포함된 최종 요약을 출력한다.
    final_summary = result_dict.get("final_summary", "")

    if final_summary:
        display(Markdown("## 7. 최종 요약"))
        display(Markdown(final_summary))


# ============================================================
# 최종 출력
# ============================================================

display_common_prelaunch_output_blocks(
    result_dict=checklist_result_dict,
    evidence_df=selected_evidence_for_prompt,
    max_evidence_rows=8,
)

display(Markdown("## 5. 출시 전 개발 체크리스트"))

checklist_table_df = make_checklist_table(
    checklist_result_dict,
    evidence_df=selected_evidence_for_prompt,
    drop_unknown_issues=True,
)

assert len(checklist_table_df) > 0, (
    "출력할 체크리스트 항목이 없습니다. "
    "LLM이 근거표에 없는 이슈를 만들었거나, 사용자 조건에 맞는 근거가 부족할 수 있습니다."
)

display(make_styled_checklist_table(checklist_table_df))

display_checklist_cautions(checklist_result_dict)
display_checklist_final_summary(checklist_result_dict)

## 1. 출시 전 LLM 체크리스트 생성

개발자가 입력한 조건과 유사한 Steam 인디게임의 출시 초기 D0-D30 리뷰를 바탕으로,  
출시 전 개발 단계에서 점검해야 할 항목을 체크리스트로 정리한다.

| 항목 | 값 |
|---|---|
| 입력 장르 | Action |
| 입력 가격대 | 10-20 |
| 입력 Steam 태그 | Roguelike, Pixel Graphics |
| 입력 플레이 방식 | Single-player 중심 |
| 입력 조건 직접 매칭 게임 수 | 1개 |
| 입력 조건 직접 매칭 리뷰 수 | 1개 |

체크리스트 생성에는 직접 매칭 결과뿐 아니라,  
장르·가격대·Steam 태그·플레이 방식별로 03-1에서 계산한 반복 이슈 근거도 함께 사용하였다.

## 2. 체크리스트 판단 기준

- **LLM은 우선순위를 직접 정하지 않는다.**
- 체크리스트 우선순위는 03-1에서 계산한 `priority_level`을 `fixed_priority`로 전달해 그대로 사용한다.
- 우선순위는 이슈 발생 게임 수, 조건 내 발생 비율, Steam 비추천 맥락 발생 게임 수를 기준으로 계산한 값이다.
- `High urgency`는 이전 LLM 리뷰 분류 결과에서 나온 보조 지표이며, 우선순위 산정 기준으로 직접 사용하지 않는다.
- LLM은 계산된 근거를 바탕으로 체크 질문과 확인 방법을 문장화하는 역할만 한다.

## 3. 조건별 반복 이슈 근거표

아래 표는 03-1에서 계산한 근거 중 체크리스트 생성에 사용한 핵심 이슈만 간단히 정리한 것이다.

,이슈,해석 방향,우선순위,조건 유형,조건 값,조건 내 게임 수,이슈 발생 게임 수,조건 내 발생 비율,Steam 비추천 맥락 게임 수,LLM 부정 맥락 게임 수,High urgency 게임 수,근거 요약
0,긍정 칭찬,강화 요소,상,price_group,10-20,45,43,9560.0%,14,0,14,price_group 조건 '10-20'에서 '긍정 칭찬' 항목은 전체 45개 게임 중 43개 게임에서 반복되었다(95.6%). Steam 비추천 맥락은 14개...
1,긍정 칭찬,강화 요소,상,steam_tag,Pixel Graphics,15,14,9330.0%,3,0,2,steam_tag 조건 'Pixel Graphics'에서 '긍정 칭찬' 항목은 전체 15개 게임 중 14개 게임에서 반복되었다(93.3%). Steam 비추천 맥...
2,긍정 칭찬,강화 요소,상,play_style,Single-player 중심,124,115,9270.0%,23,0,26,play_style 조건 'Single-player 중심'에서 '긍정 칭찬' 항목은 전체 124개 게임 중 115개 게임에서 반복되었다(92.7%). Steam...
3,긍정 칭찬,강화 요소,상,genre,Action,62,57,9190.0%,14,0,13,"genre 조건 'Action'에서 '긍정 칭찬' 항목은 전체 62개 게임 중 57개 게임에서 반복되었다(91.9%). Steam 비추천 맥락은 14개 게임, S..."
4,게임플레이 루프,리스크 요소,상,price_group,10-20,45,36,8000.0%,28,33,23,price_group 조건 '10-20'에서 '게임플레이 루프' 항목은 전체 45개 게임 중 36개 게임에서 반복되었다(80.0%). Steam 비추천 맥락은 2...
5,UI/UX,리스크 요소,상,steam_tag,Pixel Graphics,15,12,8000.0%,5,10,3,steam_tag 조건 'Pixel Graphics'에서 'UI/UX' 항목은 전체 15개 게임 중 12개 게임에서 반복되었다(80.0%). Steam 비추천 맥...
6,게임플레이 루프,리스크 요소,상,genre,Action,62,46,7420.0%,28,37,26,genre 조건 'Action'에서 '게임플레이 루프' 항목은 전체 62개 게임 중 46개 게임에서 반복되었다(74.2%). Steam 비추천 맥락은 28개 게임...
7,게임플레이 루프,리스크 요소,상,steam_tag,Pixel Graphics,15,11,7330.0%,9,10,4,steam_tag 조건 'Pixel Graphics'에서 '게임플레이 루프' 항목은 전체 15개 게임 중 11개 게임에서 반복되었다(73.3%). Steam 비추...


...

- 본 체크리스트는 과거 유사 게임들의 리뷰 데이터를 기반으로 작성되었으며, 특정 게임의 성공을 보장하지 않습니다.

- 데이터 분석 결과는 참고용이며, 실제 게임의 기획 의도와 장르적 특성에 맞춰 유연하게 적용해야 합니다.

- '상' 등급의 이슈는 비추천 맥락에서 반복적으로 언급된 사항이므로 우선적으로 점검하시기 바랍니다.

## 7. 최종 요약

Action 장르와 Roguelike/Pixel Graphics 태그를 가진 1인용 게임은 게임플레이 루프와 UI/UX, 콘텐츠 분량에서 유저들의 비판이 자주 발생하므로 이에 대한 철저한 점검이 필요합니다. 반면, 그래픽과 사운드, 가격 대비 가치는 긍정적인 평가를 이끌어내는 핵심 요소이므로 이를 유지 및 강화하는 전략이 유효합니다.